In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import heapq
import seaborn as sns

from environment.environment import GraphWorldMFG_MultiGroup
from trainer.amid_trainer_graph import GraphEdgeMFG_Trainer
from solver.solver import solve_multigroup, GraphMFG_OMD_EdgeSolver_MultiGroup
from visualization.visualizationh import plot_heatmap, compute_exploitability_multigroup, plot_losses, plot_losses_line

In [2]:
import os
from pathlib import Path
import yaml
import numpy as np
import torch

# 1. Configuration Loader
def load_config(config_path="config.yaml"):
    """Load configuration safely from a YAML file relative to working directory."""
    notebook_dir = Path(os.getcwd())
    config_file = notebook_dir / config_path
    
    with open(config_file, 'r') as f:
        config = yaml.safe_load(f)
    return config

# 2. Graph Environment Factory Pattern
def create_graph_mfg_from_config(config):
    """Factory function initializing the Graph MFG environment directly from your config."""
    device = config.get("device", "cuda" if torch.cuda.is_available() else "cpu")
    print(f"Target Device: {device}")

    trainer_cfg = config["trainer"]
    
    # Extract Graph Configuration
    graph_cfg = config["graph"]
    num_nodes = graph_cfg["num_nodes"]
    
    # CRITICAL: Extract the list of lists matrix and convert to a PyTorch Tensor
    raw_matrix = graph_cfg["adjacency_matrix"]
    adjacency_matrix = torch.tensor(raw_matrix, dtype=torch.float32, device=device)
    
    # Process Group Data (Sinks and Sources are flat integers here, not grid tuples)
    groups = []
    for g in config["groups"]:
        groups.append({
            "source": int(g["source"]),
            "sink": int(g["sink"]),
            "mass": float(g["mass"])
        })
    
    solver_cfg = config["solver"]
    waiting_time = solver_cfg.get("waiting_time", 0)
    # Instantiate the Graph World Environment
    # (Matches your 'GraphWorldMFG_MultiGroup' class structure)
    env = GraphWorldMFG_MultiGroup(
        num_nodes=num_nodes,
        groups=groups,
        adjacency_matrix=adjacency_matrix,
        H = solver_cfg.get("H", None),
        device=device
    )
    
    # Create solvers for each group
    
    
    solvers = [
        GraphMFG_OMD_EdgeSolver_MultiGroup(
            env=env,
            group_idx=k,
            eta=solver_cfg["eta"],
            tau=solver_cfg["tau"],
            T=solver_cfg["T"],
            alpha=solver_cfg["alpha"],
            H=solver_cfg.get("H", None)
        )
        for k in range(env.K)
    ]

    trainer = GraphEdgeMFG_Trainer(env, solvers, leader_lr=trainer_cfg["leader_lr"])
    
    return env, solvers, trainer, config

In [3]:
config = load_config("config_graph.yaml")

env, solvers, trainer, config = create_graph_mfg_from_config(config)

# Training loop
theta_leader = torch.zeros((env.N, env.N), device=env.device)  # Initialize leader's strategy

Target Device: cpu


c:\Uzh\thesis\msc_code\thesis\environment\environment.py:16: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.A = torch.tensor(adjacency_matrix, dtype=torch.float32, device=device)


In [4]:
flows, final_flows, policies, W_cong_history, zeta_history = solve_multigroup(env, solvers, T=50, W_max=8, theta_leader=theta_leader)

In [5]:
final_flows

tensor([[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 9.8953e-01, 1.0471e-02, 0.0000e+00],
        [0.0000e+00, 9.8953e-01, 1.0471e-02, 0.0000e+00],
        [0.0000e+00, 9.8953e-01, 1.0471e-02, 0.0000e+00],
        [0.0000e+00, 9.8953e-01, 1.0471e-02, 0.0000e+00],
        [0.0000e+00, 9.8953e-01, 1.0471e-02, 0.0000e+00],
        [4.2603e-04, 9.3772e-01, 1.0897e-02, 5.0957e-02],
        [8.1801e-03, 4.3054e-05, 1.3373e-02, 9.7840e-01],
        [7.8006e-03, 5.1564e-04, 7.7110e-03, 9.8397e-01],
        [1.1032e-04, 8.2901e-03, 3.5404e-07, 9.9160e-01],
        [6.7649e-05, 7.8640e-03, 4.2402e-06, 9.9206e-01],
        [6.8173e-05, 1.1032e-04, 6.8170e-05, 9.9975e-01],
        [6.4702e-05, 6.7684e-05, 6.4667e-05, 9.9980e-01],
        [1.4678e-06, 6.8734e-05, 9.0720e-07, 9.9993e-01],
        [1.0883e-06, 6.5234e-05, 5.5658e-07, 9.9993e-01],
        [5.7267e-07, 1.4752e-06, 5.6521e-07, 1.0000e+00],
        [5.4100e-07, 1.0929e-06, 5.3642e-07, 1.0000e+00],
        [1.677

In [6]:
flows[0,:,:,0]

tensor([[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 5.1809e-02, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 9.3772e-01, 5.2357e-03, 0.0000e+00],
        [4.2603e-04, 0.0000e+00, 5.6617e-03, 0.0000e+00],
        [7.7541e-03, 4.3054e-05, 7.7110e-03, 5.1496e-03],
        [4.6557e-05, 4.7259e-04, 0.0000e+00, 1.0718e-02],
        [6.3763e-05, 7.8175e-03, 3.5404e-07, 1.8302e-02],
        [3.8862e-06, 4.6557e-05, 3.8862e-06, 4.3781e-02],
        [6.4287e-05, 6.3766e-05, 6.4284e-05, 5.3041e-01],
        [4.1480e-07, 3.9181e-06, 3.8284e-07, 9.9156e-01],
        [1.0530e-06, 6.4816e-05, 5.2436e-07, 9.9165e-01],
        [3.5367e-08, 4.1795e-07, 3.2221e-08, 9.9190e-01],
        [5.3730e-07, 1.0573e-06, 5.3299e-07, 9.9598e-01],
        [3.701

In [7]:
flows, final_flows, policies, W_cong_history = env.simulate_forward_with_policy(policies, theta_leader=theta_leader, W_max=8)

In [8]:
final_flows

tensor([[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 9.9004e-01, 9.9590e-03, 0.0000e+00],
        [0.0000e+00, 9.9004e-01, 9.9590e-03, 0.0000e+00],
        [0.0000e+00, 9.9004e-01, 9.9590e-03, 0.0000e+00],
        [0.0000e+00, 9.9004e-01, 9.9590e-03, 0.0000e+00],
        [0.0000e+00, 9.9004e-01, 9.9590e-03, 0.0000e+00],
        [3.6912e-04, 9.4074e-01, 1.0328e-02, 4.8561e-02],
        [7.4502e-03, 3.7284e-05, 1.2392e-02, 9.8012e-01],
        [7.1211e-03, 4.4645e-04, 7.0438e-03, 9.8539e-01],
        [9.3067e-05, 7.5430e-03, 2.7916e-07, 9.9236e-01],
        [5.6083e-05, 7.1739e-03, 3.3428e-06, 9.9277e-01],
        [5.6480e-05, 9.3069e-05, 5.6478e-05, 9.9979e-01],
        [5.3739e-05, 5.6108e-05, 5.3714e-05, 9.9984e-01],
        [1.1197e-06, 5.6903e-05, 6.9685e-07, 9.9994e-01],
        [8.2229e-07, 5.4141e-05, 4.2011e-07, 9.9994e-01],
        [4.3128e-07, 1.1249e-06, 4.2606e-07, 1.0000e+00],
        [4.0853e-07, 8.2543e-07, 4.0538e-07, 1.0000e+00],
        [1.161

In [9]:
flows[0,:,:,0]

tensor([[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 4.9299e-02, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 9.4074e-01, 4.9795e-03, 0.0000e+00],
        [3.6912e-04, 0.0000e+00, 5.3486e-03, 0.0000e+00],
        [7.0811e-03, 3.7284e-05, 7.0438e-03, 4.9049e-03],
        [4.0048e-05, 4.0917e-04, 0.0000e+00, 1.0173e-02],
        [5.3019e-05, 7.1338e-03, 2.7916e-07, 1.7112e-02],
        [3.0637e-06, 4.0048e-05, 3.0637e-06, 4.1392e-02],
        [5.3416e-05, 5.3021e-05, 5.3414e-05, 5.2900e-01],
        [3.2279e-07, 3.0866e-06, 2.9986e-07, 9.9233e-01],
        [7.9693e-07, 5.3816e-05, 3.9700e-07, 9.9240e-01],
        [2.5356e-08, 3.2504e-07, 2.3112e-08, 9.9262e-01],
        [4.0592e-07, 7.9991e-07, 4.0295e-07, 9.9634e-01],
        [2.606

In [12]:
print("Starting Leader Infrastructure Optimization...")
print("-" * 50)
flows_previous = flows.detach()  # Detach to avoid backprop through the entire history
W_cong_history = W_cong_history.detach()  # Detach to avoid backprop
losses = []

epochs = 25
for epoch in range(1, epochs + 1):
    
    # Execute one optimization step
    social_loss, flows_new, W_cong_history_new = trainer.train_step(flows_previous, W_cong_history, epoch)
    
    # Pass the newly generated flows as the historical footprint for the next epoch
    flows_previous = flows_new.detach() 
    W_cong_history = W_cong_history_new.detach()

    # Optional: Recompute or update W_cong_history based on flows_new if required, 
    # otherwise it continues to adapt based on the internal solve_multigroup loop.
    
    if epoch % 1 == 0:
        print(f"Epoch {epoch:02d}/{epochs} | Social Loss (Travel Time): {social_loss:.4f}")
    
    losses.append(social_loss)

print("-" * 50)
print("Training Complete!")

Starting Leader Infrastructure Optimization...
--------------------------------------------------
Adjoint vector a_T for group 0 has shape: torch.Size([20, 4, 4])
Epoch 01/25 | Social Loss (Travel Time): 6.4743
Adjoint vector a_T for group 0 has shape: torch.Size([20, 4, 4])
Epoch 02/25 | Social Loss (Travel Time): 6.4910
Adjoint vector a_T for group 0 has shape: torch.Size([20, 4, 4])
Epoch 03/25 | Social Loss (Travel Time): 6.4573
Adjoint vector a_T for group 0 has shape: torch.Size([20, 4, 4])
Epoch 04/25 | Social Loss (Travel Time): 6.4322
Adjoint vector a_T for group 0 has shape: torch.Size([20, 4, 4])
Epoch 05/25 | Social Loss (Travel Time): 6.4112
Adjoint vector a_T for group 0 has shape: torch.Size([20, 4, 4])
Epoch 06/25 | Social Loss (Travel Time): 6.3883
Adjoint vector a_T for group 0 has shape: torch.Size([20, 4, 4])
Epoch 07/25 | Social Loss (Travel Time): 6.3648
Adjoint vector a_T for group 0 has shape: torch.Size([20, 4, 4])
Epoch 08/25 | Social Loss (Travel Time): 6.342